In [170]:
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.common.exceptions import NoSuchElementException, TimeoutException, ElementNotInteractableException, ElementClickInterceptedException   
from datetime import datetime, timedelta
import traceback
import pandas as pd
import numpy as np
from time import sleep
import urllib.parse
import json
import os
import requests

pd.set_option('display.max_colwidth', None)
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 30)
pd.set_option('future.no_silent_downcasting', True)

In [171]:
def print_it(func):
    def wrapper(*args, **kwargs):
        print(f'Running {func.__name__}')
        return func(*args, **kwargs)
    return wrapper


def time_it(func):
    def wrapper(*args, **kwargs):
        print(f'    Running {func.__name__}')
        start_time = datetime.now()
        result = func(*args, **kwargs)
        end_time = datetime.now()
        execution_time = (end_time - start_time).total_seconds()
        print(f'    {func.__name__} executed in {execution_time:.4f} seconds')
        return result
    return wrapper


def reject_cookies(driver, temp=0, retries=2):
    try:
        WebDriverWait(driver, retries).until(EC.element_to_be_clickable((By.ID, 'onetrust-reject-all-handler')))
        driver.find_element(By.ID, 'onetrust-reject-all-handler').click()
    except (NoSuchElementException, TimeoutException, ElementNotInteractableException, ElementClickInterceptedException):
        if temp >= retries:
            # print(f'{retries} attempts failed to Reject cookies')
            print('Failed to reject cookies')
        else:
            reject_cookies(driver, temp + 1)


def skip_tutorial(driver, temp=0, retries=2):
    try:
        WebDriverWait(driver, retries).until(EC.element_to_be_clickable((By.CSS_SELECTOR, 'walla-button.TooltipWrapper__skip')))
        driver.find_element(By.CSS_SELECTOR, 'walla-button.TooltipWrapper__skip').click()
        driver.find_element(By.CSS_SELECTOR, 'walla-button.TooltipWrapper__skip').click()
        driver.find_element(By.CSS_SELECTOR, 'walla-button.TooltipWrapper__skip').click()
    except (NoSuchElementException, TimeoutException, ElementNotInteractableException, ElementClickInterceptedException):
        if temp >= retries:
            # print(f'{temp + 1} attempts failed to Skip the tutorial')
            print('Failed to skip the tutorial')
        else:
            skip_tutorial(driver, temp + 1)


def setup_driver(url):
    options = webdriver.ChromeOptions()
    options.set_capability("goog:loggingPrefs", {"performance": "INFO"})
    prefs = {"profile.managed_default_content_settings.images": 2}
    options.add_experimental_option("prefs", prefs)
    options.add_argument("--no-sandbox")
    options.add_argument("--headless")
    options.add_argument("--window-size=1920,1080")
    options.add_argument("--disable-extensions")
    options.add_argument('--ignore-certificate-errors')
    options.add_argument('--disable-gpu')
    options.add_argument('--disable-dev-shm-usage')
    driver = webdriver.Chrome(options=options)

    driver.get(url)

    reject_cookies(driver)
    skip_tutorial(driver)

    return driver


def get_explorer_urls(product_id=None, category_id=None, object_id=None):
    # gets the corresponding urls to wallapop listing explorer
    base = 'https://es.wallapop.com/app/search?'
    urls = []

    if product_id:
        pass ## Do later
        # url = url if (product_id == None) else url + f'keywords={urllib.parse.quote(product_id)}&'
    else:
        url = base + f'object_type_ids={object_id}&filters_source=quick_filters&latitude=40.41956&longitude=-3.69196&'
        start, increment, loops = 0, 1, 15
        current_range = increment

        for i in range(loops):
            min = start
            max = start + current_range

            if i == 0:
                urls.append((f'{min}-{max}',(url + f'max_sale_price={max}')))
            elif i + 1 == len(range(loops)):
                urls.append((f'{min+.01}-{max}',(url + f'min_sale_price={min+.01}')))
            else:
                urls.append((f'{min+.01}-{max}',(url + f'min_sale_price={min+.01}&max_sale_price={max}')))

            start += current_range
            current_range += increment + round((i)**2) # Ramps up range ~according to previous scrapes

    '''Calculate a dynamic way to search space later

    from scipy.stats import norm
    import numpy as np

    # Desired percentage (e.g., 95%)
    percentage = 0.05

    # Calculate the z-score in the log-transformed space
    z_score_log = norm.ppf(percentage, loc=3.4, scale=0.67)

    # Transform back to the original scale
    z_score_original = np.power(10, z_score_log)
    print(z_score_log)
    print(f"The z-score for {percentage*100}% in the original scale is {z_score_original}")'''

    return urls


def load_more(driver, temp=0, retries=1):
    # Clicks button that initiates infinite scroll dynamic listings
    try:
        WebDriverWait(driver, retries).until(EC.element_to_be_clickable((By.ID, 'btn-load-more')))
        driver.find_element(By.ID, 'btn-load-more').click()
    except (NoSuchElementException, TimeoutException, ElementNotInteractableException, ElementClickInterceptedException):
        if temp >= retries:
            # print(f'{temp + 1} attempts failed to Click the load button')
            print('Failed to click the load button')
            return True
        else:
            return load_more(driver, temp + 1)


def wait_listings(driver, temp=0, retries=1):
    try:
        WebDriverWait(driver, retries).until(EC.element_to_be_clickable((By.CSS_SELECTOR, 'a.ItemCardList__item')))
        return True
    except (NoSuchElementException, TimeoutException, ElementNotInteractableException, ElementClickInterceptedException):
        if temp >= retries:
            # print(f'{temp + 1} attempts failed to Find listings')
            print('Failed to find listings')
        else:
            wait_listings(driver, temp + 1)


def convert_timestamp(ts):
    return datetime.fromtimestamp(ts / 1000).strftime('%y%m%d%H')


In [172]:
def get_categories(categories=False, category_dicts=None):
    if category_dicts is None:
        category_dicts = []
    if not categories:
        categories = requests.get('https://api.wallapop.com/api/v3/categories')
        categories = categories.json()['categories']
        return get_categories(categories=categories)
    else:
        for category in categories:
            if category['subcategories']:
                formatted_category = {key: [value] for key, value in category.items() if key != 'subcategories'}
                category_dicts.append(pd.DataFrame(formatted_category))
                category_dicts = get_categories(categories=category['subcategories'], category_dicts=category_dicts)
            else:
                formatted_category = {key: [value] for key, value in category.items()}
                category_dicts.append(pd.DataFrame(formatted_category))
        return category_dicts


def find_senior_parent_id(id, parent_map):
    if pd.isna(parent_map[id]):
        return id
    else:
        return find_senior_parent_id(parent_map[id], parent_map)


def build_path(id, parent_map):
    if pd.isna(parent_map[id]):
        return str(id)
    else:
        return build_path(parent_map[id], parent_map) + '/' + str(id)


def create_category_df():
    category_df = pd.concat(get_categories()).drop(['subcategories', 'presentation', 'category_leaf_selection_mandatory', 'icon', 'vertical_id'], axis=1)
    parent_map = category_df.set_index('id')['parent_id'].to_dict()
    category_df['senior_parent_id'] = category_df['id'].apply(lambda x: find_senior_parent_id(x, parent_map))
    category_df['senior_parent_id'] = category_df['senior_parent_id'].astype(int)
    category_df['path'] = category_df['id'].apply(lambda x: build_path(x, parent_map))
    category_df['path'] = category_df['path'].str.replace('.0', '', regex=False)
    category_df['attributes'] = category_df['attributes'].apply(json.dumps)
    category_df['attributes'] = category_df['attributes'].replace('{}', np.nan)
    category_df['attributes'] = category_df['attributes'].replace('{"excluded": []}', np.nan)
    return category_df.set_index('id').reset_index()



In [173]:
# parsing functions
# import emoji
# from nltk.corpus import stopwords
# from nltk.tokenize import word_tokenize, sent_tokenize
# from langdetect import detect
# from langdetect.lang_detect_exception import LangDetectException


# LANGUAGE_MAP = {
#     'en': 'english',
#     'es': 'spanish',
#     'fr': 'french',
#     'de': 'german',
#     'it': 'italian',
#     'pt': 'portuguese',
#     'nl': 'dutch',
#     'ru': 'russian',
#     'zh-cn': 'chinese',
#     'ja': 'japanese',
# }


# def detect_language(text):
#     if text:
#         try:
#             return detect(text)
#         except LangDetectException:
#             return 'es'
#     return 'none'


# def calculate_char_metrics(text):
#     if text:
#         lowercase_char_count = 0
#         capitalized_char_count = 0
#         digit_count = 0
#         emoji_count = 0
#         non_alphanumeric_char_count = 0
#         line_break_count = 0
#         whitespace_count = 0
        
#         for char in text:
#             if char.islower():
#                 lowercase_char_count += 1
#             if char.isupper():
#                 capitalized_char_count += 1
#             if char.isdigit():
#                 digit_count += 1

#             if char in emoji.EMOJI_DATA:
#                 emoji_count += 1
#             elif not char.isalnum() and not char.isspace():
#                 non_alphanumeric_char_count += 1

#             if char == '\n':
#                 line_break_count += 1
#             elif char.isspace():
#                 whitespace_count += 1

#         metrics = {
#         'lowercase_char_count': lowercase_char_count,
#         'capitalized_char_count': capitalized_char_count,
#         'digit_count': digit_count,
#         'emoji_count': emoji_count,
#         'non_alphanumeric_char_count': non_alphanumeric_char_count,
#         'line_break_count': line_break_count,
#         'whitespace_count': whitespace_count,
#         }
#         return metrics
#     else:
#         return {
#         'lowercase_char_count': 0,
#         'capitalized_char_count': 0,
#         'digit_count': 0,
#         'emoji_count': 0,
#         'non_alphanumeric_char_count': 0,
#         'line_break_count': 0,
#         'whitespace_count': 0,
#         }


# def calculate_metrics(text, lang_code):
#     if text:
#         language = LANGUAGE_MAP.get(lang_code, 'spanish')
#         sentences = sent_tokenize(text)
#         words = [word for word in word_tokenize(text) if word.isalpha()]
#         stop_words = set(stopwords.words(language))
        
#         allcaps_word_count = 0
#         total_word_length = 0
#         unique_words = set()
#         stop_word_count = 0
        
#         for word in words:
#             if word.isupper():
#                 allcaps_word_count += 1
#             total_word_length += len(word)
#             unique_words.add(word)
#             if word.lower() in stop_words:
#                 stop_word_count += 1

#         avg_sent_len = round((len(words) / len(sentences) if sentences else 0), 2)
#         avg_word_len = round((total_word_length / len(words) if len(words) else 0), 2)
        
#         metrics = {
#             'sentence_count': len(sentences),
#             'avg_sent_len_cents': int(avg_sent_len*100),
#             'allcaps_word_count': allcaps_word_count,
#             'word_count': len(words),
#             'avg_word_len_cents': int(avg_word_len*100),
#             'unique_word_count': len(unique_words),
#             'stop_word_count': stop_word_count,
#         }
#         return metrics
#     else:
#         return {
#             'sentence_count': 0,
#             'avg_sent_len_cents': 0,
#             'allcaps_word_count': 0,
#             'word_count': 0,
#             'avg_word_len_cents': 0,
#             'unique_word_count': 0,
#             'stop_word_count': 0,
#         }

In [174]:
# Scraping functions
def verify_filepath(filepath):
    directory = os.path.dirname(filepath)
    if not os.path.exists(directory):
        os.makedirs(directory)


def save_df(filepath, df):
    assert len(df) > 0
    df = df.dropna(how='all').drop_duplicates()
    df.to_csv(filepath, index=False)


def load_dfs(category_id=None, object_id=None):
    dfs = []
    directory = f'data/{category_id}/{object_id}/'
    for filename in os.listdir(directory):
        if filename.startswith(f"{category_id}"):
            filepath = os.path.join(directory, filename)
            df = pd.read_csv(filepath)
            dfs.append(df)
    object_df = pd.concat(dfs)
    return object_df


def check_redundancy(json_df, object_df):
    global scraped_df
    json_df_comp = json_df.drop(columns=['date_last_scraped', 'date_first_seen_reserved'], errors='ignore')
    object_df_comp = object_df.drop(columns=['date_last_scraped', 'date_first_seen_reserved'], errors='ignore')
    json_df_comp = json_df_comp[object_df_comp.columns]
    if json_df_comp.apply(tuple, axis=1).isin(object_df_comp.apply(tuple, axis=1)).all():
        scraped_df = pd.concat([scraped_df, json_df])
        return True


def find_matches(object_df, json_df):
    for _, row in json_df.iterrows():
        if row['id'] in object_df['id'].values:
            idx = object_df.index[object_df['id'] == row['id']].tolist()[0]
            common_columns = object_df.columns.intersection(json_df.columns)
            if object_df.at[idx, 'date_first_seen_reserved'] != 0:
                common_columns = common_columns.difference(['date_first_seen_reserved'])

            object_df.loc[idx, common_columns] = row[common_columns]
        else:
            object_df = pd.concat([object_df, pd.DataFrame([row])], ignore_index=True)

    return object_df


def compare_scraping(json_df, category_id=None, object_id=None, range_name=None, all=False):
    global scraped_df
    global category_df
    if range_name:
        try:
            path = category_df.loc[category_df['id'] == object_id, 'path'].values[0]
            filepath = f'data/{path}/{category_id}_{object_id}_price:{range_name}.csv'
            object_df = pd.read_csv(filepath)
            if not all and check_redundancy(json_df, object_df):
                return json_df, True
            
            object_df = find_matches(object_df, json_df)          
            save_df(filepath, object_df)

        except FileNotFoundError:
            verify_filepath(filepath)
            save_df(filepath, json_df)

        scraped_df = pd.concat([scraped_df, json_df])
        return json_df, False
        
    else: # for newest
        if not all:
            object_df = load_dfs(category_id=category_id, object_id=object_id)
            if check_redundancy(json_df, object_df):
                return json_df, True
        path = category_df.loc[category_df['id'] == object_id, 'path'].values[0]
        directory = f'data/{path}/'
        urls = pd.read_csv(f'{directory}{object_id}_urls.csv')
        range_names = urls['range_name'].to_list()
        for i, range_name in enumerate(range_names):
            min_price, max_price = map(float, range_name.split('-'))
            min_price = int(min_price*100)
            max_price = int(max_price*100)
            
            if i+1 == len(range_names):
                df_splice = json_df[(json_df['price_cents'] >= min_price)]
            else:
                df_splice = json_df[(json_df['price_cents'] >= min_price) & (json_df['price_cents'] <= max_price)]
                
            if df_splice.empty:
                continue
            else:
                try:
                    filepath = f'{directory}{category_id}_{object_id}_price:{range_name}.csv'
                    object_df = pd.read_csv(filepath)
                    object_df = find_matches(object_df, df_splice)
                    save_df(filepath, object_df)
                except FileNotFoundError:
                    verify_filepath(filepath)
                    save_df(filepath, json_df)

        scraped_df = pd.concat([scraped_df, json_df])
        return json_df, False


def parse_json(response_json):
    df = pd.DataFrame(response_json['data']['section']['payload']['items'])

    # Split up shipping
    shipping_normalized = pd.json_normalize(df['shipping'])
    df = df.reset_index(drop=True)
    shipping_normalized = shipping_normalized.reset_index(drop=True)
    df = pd.concat([df, shipping_normalized], axis=1)

    # Clean up othe metrics
    df['country_code'] = df['location'].apply(lambda x: x['country_code'])
    df['num_images'] = df['images'].apply(lambda x:len(x))
    df['refurbished'] = df['is_refurbished'].apply(lambda x: list(x.values())[0])
    df['reserved'] = df['reserved'].apply(lambda x: list(x.values())[0])
    df['currency'] = df['price'].apply(lambda x: list(x.values())[1])
    df['price_cents'] = df['price'].apply(lambda x: int(list(x.values())[0]*100))

    replace_dict = {'none': 0,
                    True: 1,
                    False: 0}
    df['refurbished'] = df['refurbished'].replace(replace_dict).astype(int)
    df['reserved'] = df['reserved'].replace(replace_dict).astype(int)
    df['user_allows_shipping'] = df['user_allows_shipping'].replace(replace_dict).astype(int)
    df['item_is_shippable'] = df['item_is_shippable'].replace(replace_dict).astype(int)

    # Get time metrics
    current_time = int(datetime.now().strftime('%y%m%d%H'))
    df['date_first_seen_reserved'] = df['reserved'].apply(lambda x: current_time if x == 1 else 0)
    df['date_last_modified'] = df['modified_at'].apply(lambda ts: int(datetime.fromtimestamp(ts / 1000).strftime('%y%m%d%H')))
    df['date_first_created'] = df['created_at'].apply(lambda ts: int(datetime.fromtimestamp(ts / 1000).strftime('%y%m%d%H')))
    df['date_last_scraped'] = current_time

    df = df.drop(['images', 'shipping', 'price', 'discount', 'favorited', 'cost_configuration_id', 'taxonomy', 'location', 'favoriteable', 'created_at', 'modified_at', 'is_refurbished', 'is_favoriteable', 'category_id', 'bump'], axis=1, errors='ignore')

    # # Get language and word metrics
    # df['lang_code'] = df['description'].apply(detect_language)
    # metrics_df = df.apply(lambda row: calculate_metrics(row['description'], row['lang_code']), axis=1).apply(pd.Series)
    # metrics_df = metrics_df.astype({
    #     'allcaps_word_count': 'int',
    #     'word_count': 'int',
    #     'unique_word_count': 'int',
    #     'stop_word_count': 'int',
    #     'sentence_count': 'int',
    # })
    # df = pd.concat([df, metrics_df], axis=1)

    # # Get char metrics
    # metrics_df = df['description'].apply(calculate_char_metrics).apply(pd.Series)
    # df = pd.concat([df, metrics_df], axis=1)

    # df = df.drop(['images', 'shipping', 'price', 'discount', 'favorited', 'cost_configuration_id', 'taxonomy', 'location', 'favoriteable', 'created_at', 'modified_at', 'is_refurbished', 'is_favoriteable', 'category_id', 'bump', 'description'], axis=1, errors='ignore') # For when we activate description parsing
    
    return df


def get_response_body(driver, request_id, temp=0, retries=20):
    try:
        response_body = driver.execute_cdp_cmd("Network.getResponseBody", {"requestId": request_id})
        return response_body
    except Exception:
        if temp >= retries:
            print(f'++++++++++++++ {temp} attemps failed to get json response body')
        else:
            sleep(temp*temp/1000)
            return get_response_body(driver, request_id, temp + 1)


def get_scraping(driver, timed_out, timer=None, category_id=None, object_id=None, range_name=None, all=False):
    # global json_df
    scrapings_df = pd.DataFrame()
    redundant = False
    
    logs = driver.get_log("performance")
    while logs:
        for i, log in enumerate(logs):
            log_json = json.loads(log["message"])["message"]
            if log_json["method"] == 'Network.requestWillBeSent':
                try:
                    response_url = log_json["params"]["documentURL"]
                    if response_url.startswith("https://api.wallapop.com/api/v3/search?"):
                        request_id = log_json["params"]["initiator"]["requestId"]
                        try:
                            response_body = get_response_body(driver, request_id)
                            if not response_body or 'body' not in response_body:
                                 continue  # Skip to the next log entry if the response is empty or invalid
                            response_json = json.loads(response_body['body'])
                            json_df = parse_json(response_json)
                            json_df, redundant = compare_scraping(json_df, category_id=category_id, object_id=object_id, range_name=range_name, all=all)
                            scrapings_df = pd.concat([scrapings_df, json_df])

                        except Exception as e:
                            print(f'GET_SCRAPING ERROR: {e}')
                            print('\n')
                            print(log_json)
                            print('\n')
                            traceback.print_exc()
                            print('\n')

                except KeyError:
                    print('KEYERROR IN GET SCRAPING')
                    print(f'\n\n{log_json}')
                    
        logs = driver.get_log("performance")

    # # Print stats
    # timespent = datetime.now() - timer
    # efficiency = round(len(scrapings_df)/timespent.total_seconds(), 3)
    # print(f'    time: {timespent}')
    # print(f'    count: {len(scrapings_df)}')
    # print(f'    efficiency = {efficiency}/sec')
        
    if timed_out:
        # print('    timed_out scrape')
        return True
    elif redundant:
        # print('    redundant scrape')
        return True
    else:
        clear_explorer(driver)

        return scroll_explorer(driver, category_id=category_id, object_id=object_id, range_name=range_name, all=all)


def scroll_explorer(driver, category_id=None, object_id=None, range_name=None, all=False):
    global max_scrolls
    previous_height = driver.execute_script("return document.body.scrollHeight")
    sleep_time = 0.1
    count = 0
    timer = datetime.now()
    complete = False

    while True:
        driver.execute_script("window.scrollTo(0, document.body.scrollHeight);")
        if count == max_scrolls:
            # print(f'{object_id} Price range:{range_name} max_scrolls count reached')
            complete = get_scraping(driver, timed_out=False, timer=timer, category_id=category_id, object_id=object_id, range_name=range_name, all=all)
        try:
            WebDriverWait(driver, sleep_time).until(lambda driver: driver.execute_script("return document.body.scrollHeight") != previous_height)
            count += 1
            sleep_time /= 2
            previous_height = driver.execute_script("return document.body.scrollHeight")
        except TimeoutException:
            sleep_time += sleep_time
            if sleep_time > 1.5:
                # print(f'{object_id} {range_name} Scroll: Timed out')
                complete = get_scraping(driver, timed_out=True, timer=timer, category_id=category_id, object_id=object_id, range_name=range_name, all=all)

        if complete:
            return True


def clear_explorer(driver):
    driver.execute_script("window.scrollTo(0, 0);")
    try:
        # Find elements and delete all siblings
        element = driver.find_element(By.CSS_SELECTOR, 'a.ItemCardList__item')
        parent = element.find_element(By.XPATH, '..')
        siblings = parent.find_elements(By.XPATH, '*')
        for sibling in siblings:
            driver.execute_script("arguments[0].remove();", sibling)
    except NoSuchElementException:
        print('    No listings found to clear to clear')


def start_scraping(urls_df, urls_path, product_id=None, category_id=None, object_id=None):
    global scraped_df
    global category_df
    object_count = 0

    time_threshold = 12 ##############

    time_ago = int((datetime.now() - timedelta(hours=time_threshold)).strftime('%y%m%d%H'))
    pending_urls = urls_df[urls_df['last_checked'] < time_ago]

    for _, row in pending_urls.iterrows():
        range_start = datetime.now()
        scraped_df = pd.DataFrame()
        range_name = row['range_name']
        url = row['url']
        complete = False
        path = category_df.loc[category_df['id'] == object_id, 'path'].values[0]
        print(f'\n{path} --- Price range:{range_name} --- Checking low-to-high')
        
        driver = setup_driver(url + '&order_by=price_low_to_high')
        if wait_listings(driver):
            try:
                load_more(driver)
                complete = scroll_explorer(driver, category_id=category_id, object_id=object_id, range_name=range_name, all=True)
                driver.quit()
            except Exception as e:
                print('\n')
                traceback.print_exc()
                print('\n')
                driver.quit()
                print(f'ERROR: {e}')

            if not complete:
                print(f'{path} --- Price range:{range_name} --- Checking high-to-low')
                driver = setup_driver(url + '&order_by=price_high_to_low')
                if wait_listings(driver):
                    try:
                        load_more(driver)
                        complete = scroll_explorer(driver, category_id=category_id, object_id=object_id, range_name=range_name)
                        driver.quit()
                    except Exception as e:
                        print('\n')
                        traceback.print_exc()
                        print('\n')
                        driver.quit()
                        print(f'ERROR: {e}')

                    if not complete:
                        print(f'{path} --- Price range:{range_name} --- Checking closest')
                        driver = setup_driver(url + '&order_by=closest')
                        if wait_listings(driver):
                            try:
                                load_more(driver)
                                complete = scroll_explorer(driver, category_id=category_id, object_id=object_id, range_name=range_name)
                                driver.quit()
                            except Exception as e:
                                print('\n')
                                traceback.print_exc()
                                print('\n')
                                driver.quit()
                                print(f'ERROR: {e}')

                            if not complete:
                                print(f'{path} --- Price range:{range_name} --- Checking newest')
                                driver = setup_driver(url + '&order_by=newest')
                                if wait_listings(driver):
                                    try:
                                        load_more(driver)
                                        complete = scroll_explorer(driver, category_id=category_id, object_id=object_id, range_name=range_name)
                                        driver.quit()
                                    except Exception as e:
                                        print('\n')
                                        traceback.print_exc()
                                        print('\n')
                                        driver.quit()
                                        print(f'ERROR: {e}')
                                else:
                                    driver.quit()
                        else:
                            driver.quit()
                else:
                    driver.quit()
        else:
            driver.quit()

        # Update categories.csv
        urls_df.loc[urls_df['range_name'] == range_name, 'last_checked'] = int(datetime.now().strftime('%y%m%d%H'))
        urls_df.to_csv(urls_path, index=False)

        timespent = datetime.now() - range_start
        object_count += len(scraped_df)
        # print(f'\n{object_id} --- Price range:{range_name} --- Checked')
        print(f'    {range_name} took: {timespent}')
        print(f'    {range_name} scraped: {len(scraped_df)}')
        print(f'    {range_name} efficency: {round(len(scraped_df)/timespent.total_seconds(), 3)}/sec')


    return object_count


def check_categories(category_ids=[24200, 12579, 13100], product_id=None, reverse=False):
    global category_df
    all_scraped = 0

    for category_id in category_ids:
        category_count = 0
        category_start = datetime.now()
        category_path = 'data/categories.csv'
        category_df = create_category_df()
        category_name = category_df.loc[category_df['id'] == category_id, 'name'].values[0]
        print(f'\n=== {category_id} {category_name} CHECKING CATEGORY')

        try:
            category_file = pd.read_csv(category_path)
            pd.testing.assert_frame_equal(category_df, category_file)
        except FileNotFoundError:
            category_file = category_df
            verify_filepath(category_path)
            category_df.to_csv(category_path, index=False)

        object_type_ids = category_df[category_df['senior_parent_id'] == category_id].sort_values(by='id', ascending=False)
        for _,row in object_type_ids.iterrows():
            if not pd.isna(row['attributes']):
                attribute_path = f'data/{row['path']}/{row['id']}_attributes.json'
                verify_filepath(attribute_path)
                with open(attribute_path, 'w') as json_file:
                    json_file.write(row['attributes'])
        
        object_type_ids = object_type_ids[~object_type_ids['id'].isin(object_type_ids['parent_id'])] # Select leaves of category tree
        if reverse:
            object_type_ids = object_type_ids.sort_values(by='id', ascending=True)

        for object_id in object_type_ids['id']:
            object_start = datetime.now()
            object_name = category_df.loc[category_df['id'] == object_id, 'name'].values[0]
            path = category_df.loc[category_df['id'] == object_id, 'path'].values[0]
            print(f'\n{path} CHECKING {object_name}')

            urls = get_explorer_urls(product_id=product_id, category_id=category_id, object_id=object_id)
            try:
                urls_path = f'data/{path}/{object_id}_urls.csv'
                urls_df = pd.read_csv(urls_path)
            except FileNotFoundError:
                urls_df = pd.DataFrame(urls)
                urls_df['last_checked'] = 0
                urls_df = urls_df.rename(columns={0:'range_name', 1:'url'})
                verify_filepath(urls_path)
                urls_df.to_csv(urls_path)

            try:
                object_count = start_scraping(urls_df, urls_path, product_id=product_id, category_id=category_id, object_id=object_id)
            except Exception as e:
                print('\n')
                print(f'ERROR: {e}')
                traceback.print_exc()
                print('\n')

            timespent = datetime.now() - object_start
            category_count += object_count
            print(f'\n+++  {path} {object_name.upper()} OBJECT CHECKED')
            print(f'+++     {object_name} took: {timespent}')
            print(f'+++     {object_name} scraped: {object_count}')
            print(f'+++     {object_name} efficency: {round(object_count/timespent.total_seconds(), 3)}/sec')

        timespent = datetime.now() - category_start
        all_scraped += category_count
        print(f'\n=== {category_id} {category_name.upper()} CATEGORY CHECKED')
        print(f'===     {category_name} took: {timespent}')
        print(f'===     {category_name} scraped: {category_count}')
        print(f'===     {category_name} efficency: {round(category_count/timespent.total_seconds(), 3)}/sec')

    if all_scraped == 0:
        return True
    print('\n=== All categories checked ===\n')


def check_newest(category_ids=[24200, 12579, 13100], product_id=None, reverse=False):
    global scraped_df
    global category_df
    all_scraped = 0

    for category_id in category_ids:
        category_count = 0
        category_start = datetime.now()
        category_path = 'data/categories.csv'
        category_df = create_category_df()
        category_name = category_df.loc[category_df['id'] == category_id, 'name'].values[0]
        print(f'\n=== {category_id} {category_name} CHECKING CATEGORY')

        try:
            category_file = pd.read_csv(category_path)
            pd.testing.assert_frame_equal(category_df, category_file)
        except FileNotFoundError:
            category_file = category_df
            verify_filepath(category_path)
            category_df.to_csv(category_path, index=False)

        object_type_ids = category_df[category_df['senior_parent_id'] == category_id].sort_values(by='id', ascending=False)
        for _,row in object_type_ids.iterrows():
            if not pd.isna(row['attributes']):
                attribute_path = f'data/{row['path']}/{row['id']}_attributes'
                verify_filepath(attribute_path)
                with open(attribute_path, 'w') as json_file:
                    json_file.write(row['attributes'])
        
        object_type_ids = object_type_ids[~object_type_ids['id'].isin(object_type_ids['parent_id'])]
        if reverse:
            object_type_ids = object_type_ids.sort_values(by='id', ascending=True)

        for object_id in object_type_ids['id']:
            scraped_df = pd.DataFrame()
            object_start = datetime.now()
            object_name = category_df.loc[category_df['id'] == object_id, 'name'].values[0]
            print(f'\n{object_id} {object_name}')

            url = f'https://es.wallapop.com/app/search?object_type_ids={object_id}&category_ids={category_id}&filters_source=quick_filters&latitude=40.41956&longitude=-3.69196'            
            driver = setup_driver(url + '&order_by=newest')
            if wait_listings(driver):
                try:
                    load_more(driver)
                    scroll_explorer(driver, category_id=category_id, object_id=object_id)
                    driver.quit()
                    # print(f'{object_id} = Price range:{range_name} = newest scraped successfully')
                    # print('Scraped successfully')
                except Exception as e:
                    print('\n')
                    traceback.print_exc()
                    print('\n')
                    driver.quit()
                    # print(f'{object_id} = Price range:{range_name} = newest ERROR: {e}')
                    print(f'ERROR: {e}')
            else:
                driver.quit()

            timespent = datetime.now() - object_start
            category_count += len(scraped_df)
            # print(f'    {object_id} {object_name}')
            print(f'    {object_name} took: {timespent}')
            print(f'    {object_name} scraped: {len(scraped_df)}')
            print(f'    {object_name} efficency: {round(len(scraped_df)/timespent.total_seconds(), 3)}/sec')

        timespent = datetime.now() - category_start
        all_scraped += category_count
        print(f'===     {category_name.upper()} {category_id} --- CATEGORY CHECKED')
        print(f'===     {category_name} took: {timespent}')
        print(f'===     {category_name} scraped: {category_count}')
        print(f'===     {category_name} efficency: {round(category_count/timespent.total_seconds(), 3)}/sec')

    if all_scraped == 0:
        return True
    print('\n=== All categories checked ===\n')

In [175]:
categories = [24200]
scraped_df = pd.DataFrame()
category_df = pd.DataFrame()
loop_count = 0
max_scrolls = 1
while True:
    try:
        if loop_count:
            check_newest(categories)
        else:
            # max_scrolls = 1
            check_categories(categories)
            # if check_categories(categories):
            #     now = datetime.now()
            #     next_hour = (now + timedelta(hours=1)).replace(minute=0, second=0, microsecond=0)
            #     time_to_sleep = (next_hour - now).total_seconds()
            #     sleep(time_to_sleep)
        loop_count += 1
        print(f'\n\n\n LOOP {loop_count} COMPLETE\n\n\n')
    except Exception:
        print('error outside')
        print('\n')
        traceback.print_exc()
        print('\n')
        break


=== 24200 Technology & electronics CHECKING CATEGORY

24194 Other virtual reality accessories CHECKING OBJECT

+++  24200/24202/10136/24194 OTHER VIRTUAL REALITY ACCESSORIES OBJECT CHECKED
+++     Other virtual reality accessories took: 0:00:00.001341
+++     Other virtual reality accessories scraped: 0
+++     Other virtual reality accessories efficency: 0.0/sec

24193 Other photo items CHECKING OBJECT

+++  24200/10200/24193 OTHER PHOTO ITEMS OBJECT CHECKED
+++     Other photo items took: 0:00:00.000780
+++     Other photo items scraped: 0
+++     Other photo items efficency: 0.0/sec

24192 Other image items CHECKING OBJECT

+++  24200/10206/24192 OTHER IMAGE ITEMS OBJECT CHECKED
+++     Other image items took: 0:00:00.001084
+++     Other image items scraped: 0
+++     Other image items efficency: 0.0/sec

24191 Other photo accessories CHECKING OBJECT

+++  24200/10200/24180/24191 OTHER PHOTO ACCESSORIES OBJECT CHECKED
+++     Other photo accessories took: 0:00:00.000996
+++     Ot